In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from analysis import SeaTurtleID2022, TurtlewatchEgypt

from animal_re_id.config import load_paths
from animal_re_id.models import turtle_detector

warnings.filterwarnings("ignore")

In [ ]:
# name = "SeaTurtleID2022"
name = "TurtlewatchEgypt"
local_path = "../../config/paths.json"
paths = load_paths(name, local_path=local_path)

if name == "TurtlewatchEgypt":
    analysis = TurtlewatchEgypt(paths)
elif name == "SeaTurtleID2022":
    analysis = SeaTurtleID2022(paths)

In [ ]:
df = analysis.get_summary()
df

In [ ]:
years = np.sort(df["year"].unique())
counts_all = {}
counts_known = {}
for year in years:
    database = df[df["year"] < year]
    query = df[df["year"] == year]

    counts_all[year] = len(query)
    counts_known[year] = query["identity"].isin(database["identity"]).sum()

In [ ]:
n_plots = 20

model = turtle_detector()
idx = np.linspace(0, len(analysis.dataset)-1, n_plots).astype(int)

for i in idx:
    img = analysis.make_prediction(model, i)
    img.resize((1280, 960)).save(f"{name}_prediction_{i:04}.png")

In [ ]:
plot_df = pd.DataFrame(
    {
        "all encounters": counts_all,
        "known encounters": counts_known,
    }
)
plot_df["new encounters"] = plot_df["all encounters"] - plot_df["known encounters"]
n_encounters = plot_df["all encounters"].sum()
display(plot_df)
display(plot_df.sum())
print(df["identity"].nunique(), len(analysis.dataset))

In [ ]:
cols = plot_df.columns

plot_df[cols].plot(kind="bar", figsize=(6, 3), width=0.8)
plt.xticks(np.arange(-0.5, len(plot_df), 1), minor=True)
plt.grid(which="minor", axis="x", alpha=0.6)
plt.grid(which="major", axis="y", alpha=0.6)
plt.gca().set_axisbelow(True)
plt.xlim([-0.5,len(plot_df)-0.5])
plt.savefig(f"{name}_counts.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
def bin_values(s):
    return pd.cut(s, bins=[-1, 0, 1, 2, 3, 4, np.inf], labels=["0", "1", "2", "3", "4", "5+"])


binned = {}
for c in ["heads", "front flippers", "hind flippers", "total"]:
    binned[c] = bin_values(df[c]).value_counts().sort_index() / n_encounters

plot_df = pd.DataFrame(binned)
plot_df.plot(kind="bar", figsize=(6, 3), width=0.8)
plt.xticks(np.arange(-0.5, len(plot_df), 1), minor=True)
plt.xticks(rotation=0)
plt.grid(which="minor", axis="x", alpha=0.6)
plt.grid(which="major", axis="y", alpha=0.6)
plt.gca().set_axisbelow(True)
plt.xlim([-0.5,len(plot_df)-0.5])
plt.xlabel("number of segmentation masks per encounter")
plt.ylabel("fraction of encounters")
plt.ylim([0, 1])
plt.savefig(f"{name}_segmentation.png", dpi=300, bbox_inches="tight")
plt.show()